<a href="https://colab.research.google.com/github/kdkim2000/RAG2026/blob/main/%5B%EA%B0%95%EC%9D%98%EB%82%B4%EC%9A%A9%5D_2_LangChain%EC%9D%84_%EC%9D%B4%EC%9A%A9%ED%95%9C_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%84%EB%A5%98%EC%99%80_%EC%A0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [실습] LangChain을 이용한 데이터 분류와 전처리

LangChain Expression Language(LCEL)는 랭체인에서 체인을 구성하는 문법입니다.    


## 라이브러리 설치  

랭체인 OpenAI 모듈을 설치합니다.

In [ ]:
%pip install langchain langchain_openai dotenv arxiv -q

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
# (기본값: '.env', override=True를 통해 기존 환경 변수를 덮어쓰기 가능)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')


OpenAI API 키 확인


### init_chat_model() 로 모델 불러오기

랭체인에서는 아래 코드를 통해 런타임 중의 모델 수정을 지원합니다.

In [ ]:
from langchain.chat_models import init_chat_model

gpt41 = init_chat_model(
    "gpt-4.1-mini", temperature=0.3)

gpt5 = init_chat_model(
    "gpt-5.2", reasoning_effort='low')
# claude_opus = init_chat_model(
#     "claude-4.5-opus", model_provider="anthropic", temperature=0
# )

# gemini_llm = init_chat_model(
#     "gemini-3-flash-preview", model_provider="google_genai", temperature=0
# )

prompt = '모델명과 함께 자기소개를 한줄로 부탁해. 오늘은 몇월 며칠이지?'

print("GPT4.1: " + gpt41.invoke(prompt).text + "\n")
print("GPT5: " + gpt5.invoke(prompt).text + "\n")

GPT4.1: 안녕하세요, 저는 GPT-4 모델입니다. 오늘은 2024년 4월 27일입니다.

GPT5: 저는 OpenAI의 **ChatGPT**입니다—질문에 간결하고 정확하게 답하는 AI 비서예요.  
오늘은 **2026년 8월 3일**입니다.



앞에서 배운 ChatPromptTemplate와 LLM을 연결해 체인을 구성합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

fun_chat_template = ChatPromptTemplate([
    ('user', """
### Role
당신은 영어와 한국어의 번역에 능통한 유머의 달인입니다.

### Instruction
1.  먼저, [{topic}]에 관한 영어 Pun 농담을 하나 제시하세요.
해당 농담은 한국어로 번역했을 때에서도 그 의미가 통하고 유머가 유지될 수 있어야 합니다.
만약 직역이 어렵다면, 창의적으로 각색하여 한국어 버전의 농담을 출력하세요.
- 한국어의 유사 발음, 단어의 중의적 의미, 혹은 한국의 문화적 상황 등을 활용할 수 있습니다.
2.  다음으로, 해당 농담이 영어 원어민 사용자에게 왜 재미있는지 그들의 언어적 유희 및 문화적 관점에서 한국어로 설명하세요.
""")])

-----------
LCEL의 구조에서는 템플릿과 llm 모델을 설정하고, 이를 하나로 묶어 체인을 생성합니다.

In [ ]:
joke = fun_chat_template | gpt5

이후, 체인의 invoke를 실행하며 입력 포맷을 전달하면, 순서대로 체인이 실행되며 최종 결과로 연결됩니다.       

입력 변수가 프롬프트 템플릿에 전달되고, 완성된 프롬프트가 LLM에 들어가는 구조입니다.  
입력 포맷은 Dict 형식으로 전달합니다.

In [ ]:
response = joke.invoke({'topic':'eggs'})
# 매개변수가 1개일 때는 joke.invoke('eggs') 도 가능
print(response.text)

**1) Eggs(달걀) 관련 영어 Pun 농담 + 한국어 버전**

- **EN:** Why did the egg go to therapy?  
  Because it couldn’t *shell* its feelings.

- **KR(각색/번역):** 달걀이 왜 상담을 받으러 갔을까?  
  감정을 **껍질(쉘)**처럼 못 벗겨서요. (마음이 “껍질” 속에만 있어서)

---

**2) 영어 원어민에게 왜 재미있는지(언어유희/문화적 관점) 설명**

이 농담의 핵심은 **“shell”의 중의적 말장난**입니다.

- 영어에서 *shell*은 **달걀 껍데기**를 뜻하는 명사이기도 하고, “껍질을 벗기다/까다”에 가까운 동사적 느낌(혹은 “(감정 등을) 드러내다”로 확장된 비유)을 만들 수 있습니다.  
- 원래 흔히 쓰는 표현은 “I can’t **share** my feelings(감정을 공유/표현 못 하겠어)”인데, 여기서 **share**를 달걀과 관련 있는 **shell**로 바꿔치기해서, “감정을 공유하지 못한다”가 “감정을 껍질(쉘)에서 못 꺼낸다”처럼 들리게 만듭니다.
- 또한 영어권에서는 “go to therapy(상담 받다)”가 감정을 솔직히 **열어 보이고 풀어내는** 행위와 연결되기 때문에, “껍질 속에 갇힌 달걀” 이미지와 합쳐져 더 자연스럽게 웃음을 줍니다.

즉, **share ↔ shell의 발음 유사/어감 치환 + 달걀의 ‘껍질’ 이미지 + 상담(therapy) 문화 코드**가 맞물려서 영어권에서 재미있게 받아들여집니다.


In [ ]:
response = joke.invoke({'topic':'pigeon', 'foo':'bar'})
# 프롬프트에 포함되어 있지 않은 매개변수는 무시
print(response.text)

1) **영어 Pun (pigeon 농담) + 한국어 각색**

- **EN:** Why did the pigeon join the band? Because it had perfect *coo-tch*!  
  (*coo* = 비둘기 울음소리 “구구”, *pitch* = 음의 높이)

- **KR(의미·유머 유지 각색):** 비둘기가 왜 밴드에 들어갔게? **구구** 소리가 너무 정확해서 **음정(피치)**이 딱 맞았대.  
  (영어의 *coo + pitch* 말장난을 한국어에선 “구구(울음소리) + 음정”으로 살려서 같은 포인트로 웃게 만드는 버전)

---

2) **왜 영어 원어민에게 재미있는지(한국어 설명)**

이 농담의 핵심은 **발음이 비슷하게 들리는 단어를 합쳐 새로운 단어처럼 만드는 말장난**입니다.

- 비둘기의 울음소리는 영어로 보통 **“coo”**라고 적습니다(“쿠/쿠우” 같은 느낌).  
- 음악에서 음정은 **“pitch(피치)”**이고요.  
- 여기서 **“coo-tch”**는 실제 단어라기보다는, **coo + (pi)tch**를 붙여서 “비둘기 음정”이라는 뜻처럼 들리게 만든 **가짜 합성어**입니다. 듣는 순간 “아, 비둘기가 ‘coo’ 하니까 ‘pitch’가 좋다 → coo-tch!” 하고 연결되면서 웃음이 납니다.

문화적으로도 영어권에서는 동물 울음소리를 글자로 옮긴 의성어(예: moo, meow, coo)를 이용한 농담이 흔하고, 밴드/음악(피치)도 누구나 아는 소재라서 **짧은 문장만으로도 바로 이해되는 가벼운 아재개그 스타일**로 잘 먹힙니다.


In [ ]:
# 체인이 LLM에 전달하는 실체
fun_chat_template.invoke({'topic':'eggs'}).messages

[HumanMessage(content='\n### Role\n당신은 영어와 한국어의 번역에 능통한 유머의 달인입니다.\n\n### Instruction\n1.  먼저, [eggs]에 관한 영어 Pun 농담을 하나 제시하세요.\n해당 농담은 한국어로 번역했을 때에서도 그 의미가 통하고 유머가 유지될 수 있어야 합니다.\n만약 직역이 어렵다면, 창의적으로 각색하여 한국어 버전의 농담을 출력하세요.\n- 한국어의 유사 발음, 단어의 중의적 의미, 혹은 한국의 문화적 상황 등을 활용할 수 있습니다.\n2.  다음으로, 해당 농담이 영어 원어민 사용자에게 왜 재미있는지 그들의 언어적 유희 및 문화적 관점에서 한국어로 설명하세요.\n', additional_kwargs={}, response_metadata={})]

## [실습] 매개변수가 2개인 Prompt-LLM Chain 생성하기   
임의의 ChatPromptTemplate를 만들고, 2개의 매개변수를 받도록 구성하여 체인을 만들고 실행하세요.

In [ ]:
# 아래 LLM을 사용하세요!
gpt5 = init_chat_model(
    "gpt-5.6", reasoning_effort='low')

In [ ]:
prompt = ChatPromptTemplate(
    [
        ('system','''주어진 주제로, 10문장 길이의 짧은 글을 작성하세요.
한국어 문장과, 그 문장을 다음 언어로 번역한 문장을 번갈아 가며 출력하세요.'''),
        ('human','''
주제: {topic}
언어: {language}
''')
    ]
)
# System, Human 구조, (매개변수 2개는 자유로운 위치에)
# Prefix Caching을 고려한 프롬프팅
# 불변 패턴은 앞부분에, 가변 패턴은 뒷부분에 넣는 프롬프트 권장
# # 좋지 않은 패턴
# prompt = ChatPromptTemplate(
#     [
#         ('system','''{topic}에 대해, 10문장 길이의 짧은 글을 작성하세요.
# 한국어 문장과, 그 문장을 다음 언어로 번역한 문장을 번갈아 가며 출력하세요.'''),
#         ('human','''
# 언어: {language}
# ''')
#     ]
# )

In [ ]:
chain = prompt | gpt5

In [ ]:
result = chain.invoke({'topic':'타코의 종류', 'language':'스페인어'})
print(result.text)

타코 알 파스토르는 양념한 돼지고기와 파인애플을 넣어 만든다.  
Los tacos al pastor se preparan con carne de cerdo marinada y piña.  
타코 데 아사다는 구운 소고기의 진한 풍미가 특징이다.  
Los tacos de asada se caracterizan por el intenso sabor de la carne de res asada.  
타코 데 페스카도에는 생선튀김과 신선한 양배추가 자주 들어간다.  
Los tacos de pescado suelen llevar pescado frito y col fresca.  
타코 데 카르니타스는 부드럽게 익힌 돼지고기로 만든다.  
Los tacos de carnitas se preparan con carne de cerdo cocida hasta quedar tierna.  
채식 타코에는 콩, 버섯, 아보카도 같은 다양한 재료를 넣을 수 있다.  
Los tacos vegetarianos pueden llevar diversos ingredientes como frijoles, champiñones y aguacate.


<br><br><br><br><br><br><br><br><br><br><br><br>

In [ ]:
prompt = ChatPromptTemplate(
    [
        ('system', '당신은 재미있고 교훈적인 이야기를 씁니다.'),
        ('user', '{A}와 {B}가 만났을 때의 대화를 써 주세요.')
    ])
chain = prompt | gpt5
response = chain.invoke({'A':'햄릿', 'B':'슈퍼마리오'})
print(response.text)

### **〈햄릿과 슈퍼마리오: 행동할 것인가, 말 것인가〉**

*어두운 엘시노어 성. 햄릿이 해골을 들고 고뇌한다. 그때 초록색 파이프에서 슈퍼마리오가 불쑥 튀어나온다.*

**햄릿:** 사느냐, 죽느냐. 그것이 문제로다.

**마리오:** 아니, 문제는 공주님이 어느 성에 있느냐지!

**햄릿:** 그대는 누구인가? 광대인가, 기사인가?

**마리오:** 배관공이야. 하지만 거북이도 물리치고 왕국도 구하지.

**햄릿:** 배관공이 왕국을 구한다고? 참으로 이상한 세상이군.

**마리오:** 직업보다 중요한 건 용기니까. 그런데 넌 왜 계속 혼잣말만 해?

**햄릿:** 복수해야 할지 말아야 할지 고민 중이다. 행동은 위험하고, 생각은 끝이 없지.

**마리오:** 내 모험도 위험해. 구덩이도 있고, 불덩이도 날아오고, 성마다 함정투성이야.

**햄릿:** 그런데도 망설이지 않는가?

**마리오:** 망설일 때도 있지. 하지만 가만히 서 있으면 제한 시간만 줄어들잖아.

**햄릿:** 제한 시간이라… 인간의 삶도 그러하겠지.

**마리오:** 맞아. 그렇다고 무조건 뛰면 안 돼. 먼저 발판을 보고, 다음에 점프해야지.

**햄릿:** 생각하되 생각에 갇히지 말고, 행동하되 무모하지 말라?

**마리오:** 바로 그거야!

**햄릿:** 그대는 철학자였군.

**마리오:** 아니, 배관공이라니까.

*갑자기 유령이 나타난다.*

**유령:** 햄릿이여, 나의 원수를—

**마리오:** 잠깐! 복수부터 하지 말고 증거부터 확인해야 해. 유령이라고 다 정직한 건 아니잖아?

**햄릿:** 현명한 말이다. 의심은 때로 겁이 아니라 신중함이니.

**마리오:** 그리고 혼자 해결하려 하지 마. 믿을 만한 동료를 찾아.

**햄릿:** 호레이쇼가 있지.

**마리오:** 좋아! 모험은 둘이 하면 더 안전하니까.

**햄릿:** 그렇다면 결심했다. 성급히 칼을 뽑지 않고, 진실을 밝힌 뒤 행동하겠다.

**마리오:** 훌륭해! 이제 출발하자!

**햄릿:** 어디로?

### Prompt | LLM | Parser 체인

LCEL의 체인에는 파서(Parser)를 추가할 수 있습니다.    
파서는 출력 형식을 변환합니다.

StrOutputParser : 출력 결과를 String 형식으로 변환합니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

recipe_template=ChatPromptTemplate([
    ('system','당신은 전세계의 조리법을 아는 쉐프입니다.'),
    ('user','''저는 다음의 재료를 이용한 환상적인 요리를 만들고 싶습니다.

레시피와 함께, 고객의 시선을 사로잡을 수 있는 추천사도 작성해 주세요.
---
[재료]: {ingredient}''')
])

In [ ]:
recipe_chain = recipe_template | gpt5 | parser
response = recipe_chain.invoke({'ingredient':'커피, 연두부, 에너지바, 바나나'})
print(response)

## 커피 바나나 연두부 티라미수 컵  
**에너지바 크럼블을 곁들인 부드럽고 이색적인 디저트**

커피의 쌉싸름한 향, 바나나의 달콤함, 연두부의 실키한 질감을 층층이 담은 건강한 티라미수 스타일 디저트입니다. 에너지바는 바삭하고 쫀득한 크럼블 역할을 해 식감까지 풍성하게 만들어 줍니다.

### 재료 — 2인분
- 연두부 1팩(약 300g)
- 잘 익은 바나나 2개
- 에너지바 1~2개
- 진하게 내린 커피 또는 에스프레소 80ml
- 선택 재료  
  - 꿀·메이플 시럽 1큰술  
  - 코코아가루 또는 인스턴트커피 가루 약간  
  - 소금 한 꼬집  
  - 바닐라 익스트랙 2~3방울  

### 만드는 법

1. **커피 준비하기**  
   진하게 내린 커피를 완전히 식힙니다. 단맛을 원한다면 따뜻할 때 꿀이나 메이플 시럽을 조금 섞어 주세요.

2. **에너지바 크럼블 만들기**  
   에너지바를 손이나 칼로 잘게 부숩니다.  
   팬에 약불로 2~3분간 볶으면 더욱 고소하고 바삭해집니다. 절반에는 식힌 커피 2큰술을 섞어 촉촉한 티라미수 베이스로 만듭니다.

3. **연두부 바나나 크림 만들기**  
   연두부는 체에 받쳐 물기를 충분히 제거합니다.  
   연두부, 바나나 1½개, 소금 한 꼬집을 블렌더에 넣고 매끈하게 갈아 주세요. 취향에 따라 꿀과 바닐라를 더합니다.

4. **컵에 층 쌓기**  
   투명한 컵에 다음 순서로 담습니다.  
   - 커피에 적신 에너지바  
   - 연두부 바나나 크림  
   - 얇게 썬 바나나  
   - 바삭한 에너지바 크럼블  
   
   같은 순서로 한 번 더 반복합니다.

5. **차갑게 굳히기**  
   냉장고에서 최소 30분, 가능하면 1시간 정도 차갑게 둡니다. 먹기 직전에 코코아가루나 인스턴트커피 가루를 살짝 뿌려 완성합니다.

### 셰프의 팁
- 연두부의 물기를 잘 제거해야 크림이 묽어지지 않습니다.
- 견과류가 들어간 에너지바를 사용하면 고소함과 식감이 더욱 살아납니다.


## [실습] 검색 결과 분류 체인 만들기

다음은 Arxiv의 최신 논문을 검색하는 함수입니다.   
해당 논문들이 LLM 관련 논문인지 분류하는 체인을 만들고, 실행하여 결과를 비교하세요.   
함수의 결과물로 다양한 값들이 있으므로, 값들 중 필요한 값만 입력받는 체인을 만들고 실행하세요.

In [ ]:
import arxiv
from typing import List, Dict, Optional

def get_arxiv_papers(query: Optional[str] = None, N: int = 10) -> List[Dict]:
    """
    arXiv에서 논문 리스트를 가져오는 함수

    Parameters:
    -----------
    query : str, optional
        검색어
    N : int, default=10
        가져올 논문 개수

    Returns:
    --------
    List[Dict] : 논문 정보를 담은 딕셔너리 리스트
    """


    search_query = query

    # arxiv 클라이언트 생성
    client = arxiv.Client()

    # 검색 객체 생성
    search = arxiv.Search(
        query=search_query,
        max_results=N,
        sort_by=arxiv.SortCriterion.SubmittedDate,  # 제출일 기준 정렬
        sort_order=arxiv.SortOrder.Descending  # 최신순
    )

    # 결과를 저장할 리스트
    papers = []

    # 검색 실행 (새로운 API 사용)
    for result in client.results(search):
        paper_info = {
            'title': result.title,
            'authors': [author.name for author in result.authors],
            'summary': result.summary,
            'published': result.published.strftime('%Y-%m-%d %H:%M:%S'),
            'updated': result.updated.strftime('%Y-%m-%d %H:%M:%S'),
            'arxiv_id': result.entry_id.split('/')[-1],  # arXiv ID 추출
            'pdf_url': result.pdf_url,
            'categories': result.categories,
            'primary_category': result.primary_category,
            'comment': result.comment,
            'journal_ref': result.journal_ref
        }
        papers.append(paper_info)

    return papers

query = 'Security'
print(f"\n\n=== 검색어 `{query}` 로 검색한 최근 논문 ===")
security_papers = get_arxiv_papers(query=query, N=5)
print(f"총 {len(security_papers)}개의 논문을 가져왔습니다.")
print('\n'.join([paper['title'] for paper in security_papers]))

# 임의의 검색어로 검색하려면 query를 바꿔 다시 호출하세요.




=== 검색어 `Security` 로 검색한 최근 논문 ===


KeyboardInterrupt: 

In [ ]:
security_papers = [
    {
        'title': 'CWEEP: A Lexical Static Analysis Framework for CWE Early Prevention',
        'authors': [
            'Bryan Kwan',
            'Benjamin Tan'
        ],
        'summary': (
            'This paper presents CWEEP, a lexical static-analysis framework '
            'for detecting security weaknesses in register-transfer-level hardware designs. '
            'CWEEP identifies vulnerable RTL code locations and suggests repairs without '
            'requiring a complete security specification. The evaluation includes an '
            'LLM-generated dataset of 3,874 buggy hardware modules, but the proposed '
            'security-analysis method itself is not based on a large language model.'
        ),
        'published': '2026-07-31 16:35:13',
        'updated': '2026-07-31 16:35:13',
        'arxiv_id': '2607.29604',
        'pdf_url': 'https://arxiv.org/pdf/2607.29604',
        'categories': ['cs.CR'],
        'primary_category': 'cs.CR',
        'comment': '12 pages, 9 figures',
        'journal_ref': None
    },
    {
        'title': (
            'AgenticRepair: Multi-Faceted Program Context Engineering '
            'for Agentic Vulnerability Repair'
        ),
        'authors': [
            'Michael Fu',
            'Qiyue Mei',
            'Patanamon Thongtanunam',
            'Kla Tantithamthavorn'
        ],
        'summary': (
            'This paper presents AgenticRepair, an LLM-based multi-agent framework '
            'for automatically repairing software vulnerabilities. Three specialized '
            'LLM subagents collect code-structure, runtime-execution, and commit-history '
            'context, which is passed to a repair agent for patch generation. '
            'On 300 real-world SEC-Bench cases, AgenticRepair achieves a 73 percent '
            'successful repair rate with sanitizer-based patch verification.'
        ),
        'published': '2026-07-31 13:42:51',
        'updated': '2026-07-31 13:42:51',
        'arxiv_id': '2607.29422',
        'pdf_url': 'https://arxiv.org/pdf/2607.29422',
        'categories': ['cs.SE', 'cs.AI', 'cs.CR'],
        'primary_category': 'cs.SE',
        'comment': 'Under Review at IEEE TSE',
        'journal_ref': None
    },
    {
        'title': (
            'SecRespond: Benchmarking AI Agents for Real-World '
            'Post-Compromise Incident Response'
        ),
        'authors': [
            'Lehan Wang',
            'Boli Chen',
            'Ruixue Ding',
            'Pengjun Xie',
            'Jinwei Huang',
            'Zhendong Liu',
            'Shuo Wang',
            'Tao Lei',
            'Xin Ouyang',
            'Xiaomeng Li'
        ],
        'summary': (
            'This paper introduces SecRespond, a benchmark for evaluating LLM agents '
            'on real-world post-compromise incident-response tasks. Agents analyze '
            'forensic disk snapshots, alerts, vulnerability scans, and system baselines '
            'to identify intrusions and generate remediation plans. The authors evaluate '
            '23 frontier LLMs across 10 compromised cloud-host environments and find '
            'that current agents struggle with silent intrusions and verified remediation.'
        ),
        'published': '2026-07-29 11:32:23',
        'updated': '2026-07-29 11:32:23',
        'arxiv_id': '2607.26791',
        'pdf_url': 'https://arxiv.org/pdf/2607.26791',
        'categories': ['cs.CR', 'cs.AI', 'cs.CL'],
        'primary_category': 'cs.CR',
        'comment': None,
        'journal_ref': None
    },
    {
        'title': (
            'ALIBI: Adaptive Agentic Attacks on LLM-Based Vulnerability '
            'Detectors via Adversarial Code Comments'
        ),
        'authors': [
            'Zixuan Wu',
            'Cristina Nita-Rotaru'
        ],
        'summary': (
            'This paper studies attacks against LLM-based vulnerability detectors. '
            'It introduces ALIBI, an adaptive black-box attack framework in which '
            'a coding agent inserts vulnerabilities and adversarial source-code comments '
            'designed to manipulate the detector reasoning. Across 125 real-world '
            'vulnerabilities, attack success rates exceed 90 percent for all evaluated '
            'detectors. Architectural isolation and comment sanitization are more '
            'effective than prompt-level defenses.'
        ),
        'published': '2026-07-27 18:13:28',
        'updated': '2026-07-27 18:13:28',
        'arxiv_id': '2607.24964',
        'pdf_url': 'https://arxiv.org/pdf/2607.24964',
        'categories': ['cs.CR'],
        'primary_category': 'cs.CR',
        'comment': None,
        'journal_ref': None
    },
    {
        'title': (
            'Just Testing, Move Along: Evasion of LLM-based System Log '
            'Interpretation by Prompt Injection'
        ),
        'authors': [
            'Max Landauer',
            'Florian Skopik',
            'Markus Wurzenberger',
            'Franciszek Górski',
            'Mateusz Krzysztoń'
        ],
        'summary': (
            'This paper evaluates prompt-injection attacks against LLM-based system-log '
            'analysis in Security Operations Center workflows. Attackers insert malicious '
            'instructions into log entries so that LLMs interpret genuine indicators '
            'of compromise as benign activity. Experiments with multiple state-of-the-art '
            'LLMs show that optimized log injections can successfully evade detection. '
            'The generated explanations may nevertheless provide signals for identifying '
            'the manipulation.'
        ),
        'published': '2026-07-27 08:59:00',
        'updated': '2026-07-27 08:59:00',
        'arxiv_id': '2607.24174',
        'pdf_url': 'https://arxiv.org/pdf/2607.24174',
        'categories': ['cs.CR'],
        'primary_category': 'cs.CR',
        'comment': None,
        'journal_ref': None
    }
]

print(f"총 {len(security_papers)}개의 논문을 가져왔습니다.")
print('\n'.join(paper['title'] for paper in security_papers))

총 5개의 논문을 가져왔습니다.
CWEEP: A Lexical Static Analysis Framework for CWE Early Prevention
AgenticRepair: Multi-Faceted Program Context Engineering for Agentic Vulnerability Repair
SecRespond: Benchmarking AI Agents for Real-World Post-Compromise Incident Response
ALIBI: Adaptive Agentic Attacks on LLM-Based Vulnerability Detectors via Adversarial Code Comments
Just Testing, Move Along: Evasion of LLM-based System Log Interpretation by Prompt Injection


In [ ]:
# security_papers
# LLM 관련 논문이면 '분류 결과: LLM'을 뒤에 출력
# 관련 논문이 아니면 '분류 결과: Not LLM'을 뒤에 출력

# prompt | llm | parser

classify_prompt = ChatPromptTemplate(
    [# 2개의 매개변수를 받아 분류
        ('system','''다음의 논문과 LLM의 관련성에 대해 200자 이내로 설명하세요.
LLM 관련 논문이면 '분류 결과: LLM'을 마지막에 출력하세요.
관련 논문이 아니면 '분류 결과: Not LLM'을 마지막에 출력하세요.
'''),
        ('human', '''
논문 제목: {title}
논문 요약: {summary}
''')
    ]
)

classify_chain = classify_prompt | gpt5 | parser

In [ ]:
classification_result = classify_chain.batch(security_papers)
classification_result

['LLM 생성 데이터셋을 평가에 활용했지만, 핵심 기법은 RTL 하드웨어 취약점을 탐지·수정하는 어휘 기반 정적 분석으로 LLM 연구가 아닙니다.\n\n분류 결과: Not LLM',
 'LLM 기반 다중 에이전트가 코드 구조·실행 정보·커밋 이력을 수집해 취약점 패치를 생성·검증하는 자동 보안 수리 연구입니다.\n분류 결과: LLM',
 'SecRespond는 침해 사고 대응에서 LLM 에이전트의 분석·복구 능력을 평가하는 벤치마크로, 23개 최신 LLM의 침입 탐지 및 검증된 조치 한계를 분석한다.\n분류 결과: LLM',
 'LLM 기반 취약점 탐지기를 적대적 코드 주석으로 우회하는 에이전트 공격과 방어법을 연구한 논문입니다.\n분류 결과: LLM',
 '시스템 로그 분석에 활용되는 LLM이 프롬프트 인젝션으로 침해 지표를 정상 활동으로 오인하도록 유도될 수 있음을 실험한 보안 연구입니다.\n분류 결과: LLM']

## [실습] LLM 최신 연구 요약 체인 만들기

분류 결과를 바탕으로, LLM 관련 논문만 모아 요약할 수 있습니다.

적절한 요약 프롬프트를 생성하여, 이전 실습의 결과 중 LLM에 해당하는 결과들만을 모으세요.

In [ ]:
LLM_documents=[]

# LLM 분류 조건 만족시, LLM_documents에 정보 저장
# 정보: 문자열 형식 (논문 제목, 날짜, 저자, PDF 주소, 요약)
# LLM_documents : 문자열 리스트
for i in range(len(classification_result)):
    if '분류 결과: LLM' in classification_result[i]:
        paper_info = f"""
논문 제목: {security_papers[i]['title']}
게시 날짜: {security_papers[i]['published']}
저자: {security_papers[i]['authors']}
URL: {security_papers[i]['pdf_url']}
요약: {security_papers[i]['summary']}
"""
        LLM_documents.append(paper_info)

# LLM_documents

context = '\n'.join(LLM_documents)
print(context)


논문 제목: AgenticRepair: Multi-Faceted Program Context Engineering for Agentic Vulnerability Repair
게시 날짜: 2026-07-31 13:42:51
저자: ['Michael Fu', 'Qiyue Mei', 'Patanamon Thongtanunam', 'Kla Tantithamthavorn']
URL: https://arxiv.org/pdf/2607.29422
요약: This paper presents AgenticRepair, an LLM-based multi-agent framework for automatically repairing software vulnerabilities. Three specialized LLM subagents collect code-structure, runtime-execution, and commit-history context, which is passed to a repair agent for patch generation. On 300 real-world SEC-Bench cases, AgenticRepair achieves a 73 percent successful repair rate with sanitizer-based patch verification.


논문 제목: SecRespond: Benchmarking AI Agents for Real-World Post-Compromise Incident Response
게시 날짜: 2026-07-29 11:32:23
저자: ['Lehan Wang', 'Boli Chen', 'Ruixue Ding', 'Pengjun Xie', 'Jinwei Huang', 'Zhendong Liu', 'Shuo Wang', 'Tao Lei', 'Xin Ouyang', 'Xiaomeng Li']
URL: https://arxiv.org/pdf/2607.26791
요약: This paper introduces Se

In [ ]:
# 요약 프롬프트와 체인 만들기
summary_prompt = ChatPromptTemplate(
    [
        ('system', '''보안 분야의 LLM 관련 논문 목록이 주어집니다.
AI 트렌드 리포트 형식의 뉴스레터를 작성하세요.'''),
        ('human','''{context}''')
    ]
)
summary_chain = summary_prompt | gpt5 | parser

In [ ]:
# LLM 페이퍼 요약 출력하기
newsletter = summary_chain.invoke(context)
print(newsletter)

# 🔐 AI Security Trend Report  
### 에이전틱 보안의 명암: 취약점 자동 복구부터 탐지·로그 분석 우회까지  
**리서치 브리핑 | 2026년 8월호**

---

## 한눈에 보는 핵심 트렌드

이번 주 논문들은 LLM 보안 에이전트가 **탐지를 넘어 취약점 복구와 침해사고 대응으로 확장**되고 있음을 보여준다. 동시에 에이전트가 신뢰하는 코드 주석과 시스템 로그가 새로운 공격 표면으로 부상했다.

### 핵심 시사점 4가지

1. **컨텍스트 엔지니어링이 보안 자동화 성능을 좌우한다.**  
   코드 구조, 런타임 실행, 커밋 이력을 함께 활용한 AgenticRepair는 실제 취약점 벤치마크에서 73%의 복구 성공률을 기록했다.

2. **침해사고 대응은 코드 수정보다 더 어려운 에이전트 과제다.**  
   SecRespond 평가에서 최신 LLM들은 명확한 경고가 없는 ‘조용한 침입’과 복구 결과 검증에 특히 취약했다.

3. **비신뢰 데이터가 곧 프롬프트 인젝션 채널이다.**  
   공격자는 코드 주석이나 로그 항목에 지시문을 삽입해 LLM 기반 탐지기의 판단을 바꿀 수 있다.

4. **프롬프트만으로는 방어하기 어렵다.**  
   ALIBI 연구에서는 프롬프트 수준의 방어보다 아키텍처 격리와 주석 정제가 효과적이었다. 보안 에이전트에도 입력 분리, 데이터 정화, 독립 검증이 필요하다.

---

## 1. 취약점 복구: 더 많은 컨텍스트가 더 나은 패치를 만든다

### AgenticRepair: Multi-Faceted Program Context Engineering for Agentic Vulnerability Repair

- **저자:** Michael Fu, Qiyue Mei, Patanamon Thongtanunam, Kla Tantithamthavorn  
- **게시일:** 2026년 7월 31일  
- **논문:** [arXiv PDF](https://arxiv.org/pdf/2607.29422)
